# **VITALS TO TEXT CONVERSION USING BIO BART**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import os

from transformers import BartTokenizer, BartForConditionalGeneration

In [ ]:
BASE_PATH = "/content/drive/MyDrive/MIMIC_Extracted"
PREP_PATH = os.path.join(BASE_PATH, "preprocessed_data")
OUTPUT_PATH = os.path.join(BASE_PATH, "biobart_output")

os.makedirs(OUTPUT_PATH, exist_ok=True)

In [ ]:
print("Loading NPZ...")

data = np.load(os.path.join(PREP_PATH, "vitals_24h_preprocessed.npz"))

X_norm = torch.tensor(data["X"], dtype=torch.float32)
X_norm = torch.nan_to_num(X_norm, nan=0.0)

patient_ids = data["ids"]
var_names = data["var_names"]

mean = torch.tensor(data["mean"], dtype=torch.float32)
std = torch.tensor(data["std"], dtype=torch.float32)

print("Patients:", len(patient_ids))
print("Variables:", var_names)
X_real = X_norm * std + mean


Loading NPZ...
Patients: 998
Variables: ['HeartRate' 'MeanBP' 'RespRate' 'Temp' 'SpO2']


In [ ]:
def vitals_to_text(vital_window):
    sentences = []

    for var, values in vital_window.items():
        values = np.array(values, dtype=np.float32)
        values = values[~np.isnan(values)]

        if len(values) == 0:
            continue

        min_v = values.min()
        max_v = values.max()
        mean_v = values.mean()

        sentences.append(
            f"{var} ranged from {min_v:.1f} to {max_v:.1f}, "
            f"with an average of {mean_v:.1f}."
        )

    return " ".join(sentences)


In [ ]:
print("\nLoading BioBART...")

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = BartTokenizer.from_pretrained("GanjinZero/biobart-v2-base")
model = BartForConditionalGeneration.from_pretrained(
    "GanjinZero/biobart-v2-base"
).to(device)

model.eval()


Loading BioBART...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/666M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/260 [00:00<?, ?it/s]

BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(85401, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(85401, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

In [ ]:
print("\nGenerating summaries...")

results = []

with torch.no_grad():
    for i in tqdm(range(len(X_real))):

        vital_window = {
            var_names[j]: X_real[i, :, j]
            for j in range(len(var_names))
        }

        base_text = vitals_to_text(vital_window)

        inputs = tokenizer(
            base_text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(device)

        summary_ids = model.generate(
            **inputs,
            max_length=128,
            num_beams=4,
            early_stopping=True
        )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True
        )

        results.append({
            "icustay_id": int(patient_ids[i]),
            "vitals_text": summary
        })


Generating summaries...


100%|██████████| 998/998 [3:46:35<00:00, 13.62s/it]


In [ ]:
df = pd.DataFrame(results)

out_file = os.path.join(OUTPUT_PATH, "vitals_biobart_text.csv")
df.to_csv(out_file, index=False)

print("\nSaved:", out_file)
print("Rows:", len(df))


Saved: /content/drive/MyDrive/MIMIC_Extracted/biobart_output/vitals_biobart_text.csv
Rows: 998
